# DKT Atlas Multi-Feature Ridge Regression

This is a separate model notebook. It does not modify `analysis.ipynb` or the first DKT surface-area notebook.

Model idea:
- use all FreeSurfer rows where `atlas == 'aparc.DKTatlas'`
- use regional `SurfArea`, `GrayVol`, and `ThickAvg` as predictors
- add participant age as another predictor
- impute missing predictor values with the training median
- standardize all predictors before fitting
- fit a Ridge regression model with cross-validated alpha selection

## Imports and Settings

In [1]:
from pathlib import Path
from urllib.error import HTTPError

import numpy as np
import pandas as pd
from rbclib import RBCPath

from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ATLAS = "aparc.DKTatlas"
MEASURES = ["SurfArea", "GrayVol", "ThickAvg"]

RBC_DATA_PATH = Path("/home/jovyan/shared/data/RBC")
TRAIN_FILE = RBC_DATA_PATH / "train_participants.tsv"
TEST_FILE = RBC_DATA_PATH / "test_participants.tsv"

FEATURE_CACHE = Path("cache/dkt_surfarea_grayvol_thickavg_features.tsv")
FS_CACHE_DIR = Path.home() / "cache" / "rbc_freesurfer"

OUTPUT_PATH = Path("results/dkt_multifeature_ridge.tsv")

# Try a broad range of regularization strengths. Larger alpha means stronger
# shrinkage of regional coefficients.
RIDGE_ALPHAS = np.logspace(-3, 5, 25)

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Load Participant Metadata

In [2]:
if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        "Expected train/test TSVs under /home/jovyan/shared/data/RBC. "
        "Run this notebook on the NeuroHackademy JupyterHub."
    )

train_data = pd.read_csv(TRAIN_FILE, sep="\t")
test_data = pd.read_csv(TEST_FILE, sep="\t")
all_data = pd.concat([train_data, test_data], ignore_index=True)

print(f"Training participants: {len(train_data)}")
print(f"Test participants: {len(test_data)}")
print("Participant metadata columns:")
print(list(all_data.columns))

all_data.head()

Training participants: 1280
Test participants: 321
Participant metadata columns:
['participant_id', 'study', 'study_site', 'session_id', 'wave', 'age', 'sex', 'race', 'ethnicity', 'bmi', 'handedness', 'participant_education', 'parent_1_education', 'parent_2_education', 'p_factor']


,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,p_factor
0,1000393599,PNC,PNC1,PNC1,1,15.583333,Male,Black,not Hispanic or Latino,22.15,Right,9th Grade,Complete primary,Complete secondary,0.589907
1,1000881804,PNC,PNC1,PNC1,1,14.916667,Male,Black,not Hispanic or Latino,21.52,Right,7th Grade,Complete secondary,Complete secondary,-0.655377
2,1001970838,PNC,PNC1,PNC1,1,17.833333,Male,Other,Hispanic or Latino,23.98,Right,11th Grade,Complete tertiary,Complete tertiary,-0.659061
3,100527940,PNC,PNC1,PNC1,1,8.250000,Male,Black,not Hispanic or Latino,NaN,Ambidextrous,1st Grade,Complete secondary,Complete primary,-0.591516
4,1006151876,PNC,PNC1,PNC1,1,21.500000,Female,Other,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,-0.377828


## Add Age Predictor

The participant TSV may use a dataset-specific age column name. This cell looks for a likely age column and makes a numeric `age_predictor` column.

In [3]:
def find_age_column(dataframe):
    """Return the most likely age column name from participant metadata."""
    exact_candidates = [
        "age",
        "Age",
        "age_years",
        "age_at_scan",
        "scan_age",
        "interview_age",
        "participant_age",
    ]

    for candidate in exact_candidates:
        if candidate in dataframe.columns:
            return candidate

    age_like = [
        col for col in dataframe.columns
        if "age" in col.lower() and pd.to_numeric(dataframe[col], errors="coerce").notna().any()
    ]
    if age_like:
        return sorted(age_like, key=len)[0]

    raise ValueError(
        "Could not find an age column. Check the printed metadata columns above "
        "and set AGE_COLUMN manually."
    )


# If the automatic choice is wrong, replace this with a string like:
# AGE_COLUMN = "your_age_column_name"
AGE_COLUMN = find_age_column(all_data)

all_data = all_data.copy()
all_data["age_predictor"] = pd.to_numeric(all_data[AGE_COLUMN], errors="coerce")

print(f"Using age column: {AGE_COLUMN!r}")
all_data[["participant_id", AGE_COLUMN, "age_predictor", "p_factor"]].head()

Using age column: 'age'


,participant_id,age,age_predictor,p_factor
0,1000393599,15.583333,15.583333,0.589907
1,1000881804,14.916667,14.916667,-0.655377
2,1001970838,17.833333,17.833333,-0.659061
3,100527940,8.250000,8.250000,-0.591516
4,1006151876,21.500000,21.500000,-0.377828


## Extract DKT Regional Features

Each region-measure pair becomes one predictor column. For example, one column might be `SurfArea__lh_superiorfrontal`. If duplicate rows occur for a region, values are summed within participant.

In [4]:
fs_root = RBCPath(
    "rbc://PNC_FreeSurfer/freesurfer",
    local_cache_dir=FS_CACHE_DIR,
)


def participant_label(participant_id):
    """Return the participant id text used in FreeSurfer filenames."""
    text = str(participant_id)
    if text.startswith("sub-"):
        text = text[4:]
    if text.endswith(".0"):
        text = text[:-2]
    return text


def load_regionsurfacestats(participant_id):
    """Load one participant's FreeSurfer region surface stats table."""
    pid = participant_label(participant_id)
    tsv_path = fs_root / f"sub-{pid}" / f"sub-{pid}_regionsurfacestats.tsv"
    with tsv_path.open("r") as f:
        return pd.read_csv(f, sep="\t")


def extract_dkt_features(participant_id):
    """Extract DKT SurfArea, GrayVol, and ThickAvg features for one participant."""
    data = load_regionsurfacestats(participant_id)
    required_cols = ["atlas", "StructName"] + MEASURES
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns for {participant_id}: {missing_cols}")

    dkt = data.loc[data["atlas"].eq(ATLAS), ["StructName"] + MEASURES].copy()
    if dkt.empty:
        raise ValueError(f"No rows found for atlas={ATLAS!r}, participant={participant_id}")

    row = {}
    for measure in MEASURES:
        dkt[measure] = pd.to_numeric(dkt[measure], errors="coerce")
        values = dkt.groupby("StructName")[measure].sum(min_count=1)
        for region_name, value in values.items():
            row[f"{measure}__{region_name}"] = value
    return row


def build_dkt_feature_table(participant_ids, feature_cache=FEATURE_CACHE):
    """Build or load the participant-by-feature DKT table."""
    if feature_cache.exists():
        print(f"Loading cached features from {feature_cache}")
        return pd.read_csv(feature_cache, sep="\t")

    feature_cache.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    total = len(participant_ids)

    for ii, participant_id in enumerate(participant_ids, start=1):
        if ii == 1 or ii % 25 == 0 or ii == total:
            print(f"Extracting {ii}/{total}: participant {participant_id}")

        row = {"participant_id": participant_id}
        try:
            row.update(extract_dkt_features(participant_id))
        except (FileNotFoundError, HTTPError, ValueError) as err:
            print(f"  Missing/unusable FreeSurfer data for {participant_id}: {err}")
        rows.append(row)

    feature_table = pd.DataFrame(rows)
    feature_table.to_csv(feature_cache, sep="\t", index=False)
    print(f"Saved feature cache to {feature_cache}")
    return feature_table

In [5]:
feature_data = build_dkt_feature_table(all_data["participant_id"].tolist())

model_data = all_data[["participant_id", "p_factor", "age_predictor"]].merge(
    feature_data,
    on="participant_id",
    how="left",
)

regional_feature_cols = [
    col for col in model_data.columns
    if any(col.startswith(f"{measure}__") for measure in MEASURES)
]
train_rows = model_data["p_factor"].notna()

# Keep only regional feature columns observed for at least one training participant.
usable_regional_feature_cols = [
    col for col in regional_feature_cols
    if model_data.loc[train_rows, col].notna().any()
]

predictor_cols = ["age_predictor"] + usable_regional_feature_cols

print(f"Regional DKT predictors extracted: {len(regional_feature_cols)}")
print(f"Usable regional DKT predictors in training data: {len(usable_regional_feature_cols)}")
print(f"Total predictors including age: {len(predictor_cols)}")

model_data.loc[:, ["participant_id", "p_factor"] + predictor_cols[:6]].head()

Extracting 1/1601: participant 1000393599
Extracting 25/1601: participant 1060532313
Extracting 50/1601: participant 1139941451
Extracting 75/1601: participant 1207361199
Extracting 100/1601: participant 1313957126
  Missing/unusable FreeSurfer data for 1342487188: [Errno 2] No such file or directory: '/home/jovyan/shared/data/RBC/repos/PNC_FreeSurfer/freesurfer/sub-1342487188/sub-1342487188_regionsurfacestats.tsv'
Extracting 125/1601: participant 1402618672
Extracting 150/1601: participant 1484543503
Extracting 175/1601: participant 1547173705
Extracting 200/1601: participant 1620557808
Extracting 225/1601: participant 1687315516
Extracting 250/1601: participant 177928631
Extracting 275/1601: participant 1833416906
Extracting 300/1601: participant 1919055552
Extracting 325/1601: participant 1998578790
  Missing/unusable FreeSurfer data for 2003542642: [Errno 2] No such file or directory: '/home/jovyan/shared/data/RBC/repos/PNC_FreeSurfer/freesurfer/sub-2003542642/sub-2003542642_region

,participant_id,p_factor,age_predictor,SurfArea__caudalanteriorcingulate,SurfArea__caudalmiddlefrontal,SurfArea__cuneus,SurfArea__entorhinal,SurfArea__fusiform
0,1000393599,0.589907,15.583333,1728.0,4526.0,5168.0,1084.0,5579.0
1,1000881804,-0.655377,14.916667,1527.0,3820.0,4278.0,714.0,6063.0
2,1001970838,-0.659061,17.833333,1594.0,4186.0,3759.0,695.0,5310.0
3,100527940,-0.591516,8.250000,2172.0,5053.0,4577.0,806.0,6061.0
4,1006151876,-0.377828,21.500000,1389.0,3922.0,3488.0,713.0,4526.0


## Cross-Validate Ridge Regression

The pipeline imputes missing predictors, standardizes every predictor, then fits Ridge regression. `RidgeCV` chooses the regularization strength from `RIDGE_ALPHAS` within each training fold.

In [6]:
X_train = model_data.loc[train_rows, predictor_cols]
y_train = model_data.loc[train_rows, "p_factor"]
X_test = model_data.loc[~train_rows, predictor_cols]

outer_cv = KFold(n_splits=min(5, len(y_train)), shuffle=True, random_state=42)
inner_cv = KFold(n_splits=min(5, len(y_train)), shuffle=True, random_state=2026)

ridge_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=RIDGE_ALPHAS, cv=inner_cv)),
])

cv_predictions = cross_val_predict(ridge_model, X_train, y_train, cv=outer_cv)

cv_metrics = pd.Series({
    "cross_validated_r2": r2_score(y_train, cv_predictions),
    "cross_validated_mae": mean_absolute_error(y_train, cv_predictions),
    "cross_validated_rmse": np.sqrt(mean_squared_error(y_train, cv_predictions)),
})

cv_metrics

/srv/conda/envs/notebook/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['SurfArea__frontalpole' 'GrayVol__frontalpole' 'ThickAvg__frontalpole']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['SurfArea__frontalpole' 'GrayVol__frontalpole' 'ThickAvg__frontalpole']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


cross_validated_r2      0.050725
cross_validated_mae     0.756722
cross_validated_rmse    0.912514
dtype: float64

## Fit Final Ridge Model and Predict Test Participants

In [7]:
final_model = ridge_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

print(f"Selected final alpha: {final_model.named_steps['ridge'].alpha_}")

predicted_test_data = test_data.copy()
predicted_test_data.loc[:, "p_factor"] = test_predictions

predicted_test_data.head()

Selected final alpha: 2154.434690031882


,participant_id,study,study_site,session_id,wave,age,sex,race,ethnicity,bmi,handedness,participant_education,parent_1_education,parent_2_education,p_factor
0,2285120424,PNC,PNC1,PNC1,1,16.166667,Female,Black,not Hispanic or Latino,22.67,Left,9th Grade,Complete secondary,Complete secondary,-0.182744
1,1854364375,PNC,PNC1,PNC1,1,16.833333,Female,White,not Hispanic or Latino,28.19,Right,10th Grade,Complete secondary,Complete primary,-0.318215
2,1448081953,PNC,PNC1,PNC1,1,19.583333,Female,White,not Hispanic or Latino,25.10,Right,Some College,Complete tertiary,Complete tertiary,-0.356040
3,1342685465,PNC,PNC1,PNC1,1,17.166667,Female,Black,not Hispanic or Latino,31.93,Right,10th Grade,Complete primary,Complete primary,-0.074728
4,3289562482,PNC,PNC1,PNC1,1,21.166667,Female,White,not Hispanic or Latino,NaN,Right,12th Grade,Complete tertiary,Complete secondary,-0.370606


## Inspect Largest Standardized Coefficients

In [8]:
coefficients = pd.Series(
    final_model.named_steps["ridge"].coef_,
    index=predictor_cols,
    name="standardized_coefficient",
)

top_coefficients = coefficients.reindex(
    coefficients.abs().sort_values(ascending=False).index
).head(30).to_frame()

top_coefficients

,standardized_coefficient
age_predictor,0.048202
ThickAvg__parahippocampal,-0.019888
GrayVol__posteriorcingulate,-0.018132
GrayVol__caudalmiddlefrontal,-0.017382
ThickAvg__supramarginal,-0.015795
ThickAvg__precentral,-0.013013
ThickAvg__pericalcarine,-0.012958
ThickAvg__caudalmiddlefrontal,-0.012860
GrayVol__superiorfrontal,-0.012125
ThickAvg__superiorfrontal,-0.012011


## Save Predictions

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predicted_test_data.to_csv(OUTPUT_PATH, sep="\t", index=False)

print(f"Saved predictions to {OUTPUT_PATH}")
OUTPUT_PATH

Saved predictions to results/dkt_multifeature_ridge.tsv


PosixPath('results/dkt_multifeature_ridge.tsv')